# Yandex Search Calling Seminar (GigaChat)

What you will learn:
1. How to connect Yandex Search API to Python tools.
2. How to build a retriever-based QA chain with sources.
3. How to expose Yandex Search as a function-calling tool for a GigaChat agent.
4. How to run an interactive chat loop via `input()`.


## Environment Setup

Before running notebook cells, make sure you have:
- `GIGACHAT_CREDENTIALS` (or `API_KEY`) for GigaChat,
- `YANDEX_API_KEY` and `YANDEX_FOLDER_ID` for Yandex Search API v2,
- a service account/API key with role `search-api.webSearch.user` and key scope `yc.search-api.execute`.


In [ ]:
# Install once if needed
# !pip install -U gigachat langchain-gigachat langgraph langchain-core python-dotenv httpx parsel


In [6]:
import os
import sys
import json
import getpass
from pathlib import Path
from operator import itemgetter
from textwrap import dedent

from dotenv import load_dotenv, find_dotenv
from IPython.display import Markdown, display

from langchain_gigachat.chat_models import GigaChat as LC_GigaChat
from langchain_gigachat.tools.giga_tool import giga_tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel
from pydantic import Field

load_dotenv(find_dotenv(usecwd=True))


True

In [7]:
def _set_env(name: str, secret: bool = True):
    if os.getenv(name):
        return
    if secret:
        os.environ[name] = getpass.getpass(f'{name}: ')
    else:
        os.environ[name] = input(f'{name}: ').strip()


if os.getenv('API_KEY') and not os.getenv('GIGACHAT_CREDENTIALS'):
    os.environ['GIGACHAT_CREDENTIALS'] = os.getenv('API_KEY')

_set_env('GIGACHAT_CREDENTIALS', secret=True)
_set_env('YANDEX_API_KEY', secret=True)
_set_env('YANDEX_FOLDER_ID', secret=False)

folder_id = os.getenv('YANDEX_FOLDER_ID', '')
if folder_id and not folder_id.startswith('b1'):
    print('Warning: YANDEX_FOLDER_ID usually starts with "b1" (folder id).')
    print('Current value looks suspicious and may be an API key/service account id.')

print('GIGACHAT_CREDENTIALS loaded:', bool(os.getenv('GIGACHAT_CREDENTIALS')))
print('YANDEX_API_KEY loaded:', bool(os.getenv('YANDEX_API_KEY')))
print('YANDEX_FOLDER_ID loaded:', bool(os.getenv('YANDEX_FOLDER_ID')))


GIGACHAT_CREDENTIALS loaded: True
YANDEX_API_KEY loaded: True
YANDEX_FOLDER_ID loaded: True


## Part 1. Built-in Yandex Search Wrapper (self-contained)

This notebook defines `YandexSearchAPIWrapper` and `YandexSearchAPIRetriever` directly, so it does not depend on external local modules.


In [8]:
import base64
import json
import re
from typing import Any, Dict, List, Literal

from langchain_core.callbacks import (
    AsyncCallbackManagerForRetrieverRun,
    CallbackManagerForRetrieverRun,
)
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.utils import get_from_dict_or_env
from pydantic import BaseModel, ConfigDict, Field, SecretStr, model_validator


class YandexSearchAPIWrapper(BaseModel):
    """Minimal Yandex Search API v2 wrapper used in this seminar."""

    folder_id: SecretStr
    api_key: SecretStr

    base_url: str = "https://searchapi.api.cloud.yandex.net"
    endpoint: str = "/v2/web/search"

    search_type: Literal["ru", "tr", "com", "kk", "be", "uz"] = "ru"
    search_region_id: int = 225
    notification_language: Literal["ru", "uk", "be", "kk", "tr", "en"] = "ru"

    sorting_rule: Literal["relevance", "document_update_time"] = "relevance"
    sorting_time_order: Literal["descending", "ascending"] = "descending"
    filtering_type: Literal["none", "moderate", "strict"] = "moderate"

    passages_count: int = Field(default=5, ge=1, le=5)
    grouping_method: Literal["flat", "deep"] = "deep"
    groups_on_page: int = Field(default=100, ge=1, le=100)
    docs_in_group: int = Field(default=1, ge=1, le=3)
    page: int = Field(default=0, ge=0)

    answer_fields: List[str] = Field(default_factory=lambda: ["url", "content"])
    passages_delimiter: str = "\n"
    request_timeout_s: float = 30.0

    model_config = ConfigDict(extra="forbid")

    @model_validator(mode="before")
    def validate_environment(cls, values: Dict) -> Dict:
        api_key = get_from_dict_or_env(values, "api_key", "YANDEX_API_KEY")
        folder_id = get_from_dict_or_env(values, "folder_id", "YANDEX_FOLDER_ID")

        # Yandex Cloud folder IDs typically start with "b1".
        # Users often paste API key ID (e.g., "aje...") into YANDEX_FOLDER_ID by mistake.
        if folder_id and not str(folder_id).startswith("b1"):
            raise ValueError(
                "YANDEX_FOLDER_ID looks invalid. Expected a folder id like `b1g...`, "
                "but got a value that does not start with `b1`."
            )

        values["api_key"] = api_key
        values["folder_id"] = folder_id
        return values

    @property
    def base_payload(self) -> Dict[str, Any]:
        search_type_map = {
            "ru": "SEARCH_TYPE_RU",
            "tr": "SEARCH_TYPE_TR",
            "com": "SEARCH_TYPE_COM",
            "kk": "SEARCH_TYPE_KK",
            "be": "SEARCH_TYPE_BE",
            "uz": "SEARCH_TYPE_UZ",
        }
        l10n_map = {
            "ru": "LOCALIZATION_RU",
            "uk": "LOCALIZATION_UK",
            "be": "LOCALIZATION_BE",
            "kk": "LOCALIZATION_KK",
            "tr": "LOCALIZATION_TR",
            "en": "LOCALIZATION_EN",
        }
        family_mode_map = {
            "none": "FAMILY_MODE_NONE",
            "moderate": "FAMILY_MODE_MODERATE",
            "strict": "FAMILY_MODE_STRICT",
        }
        sort_mode_map = {
            "relevance": "SORT_MODE_BY_RELEVANCE",
            "document_update_time": "SORT_MODE_BY_TIME",
        }
        sort_order_map = {
            "descending": "SORT_ORDER_DESC",
            "ascending": "SORT_ORDER_ASC",
        }
        group_mode_map = {
            "flat": "GROUP_MODE_FLAT",
            "deep": "GROUP_MODE_DEEP",
        }

        return {
            "query": {
                "searchType": search_type_map[self.search_type],
                "familyMode": family_mode_map[self.filtering_type],
                "page": self.page,
                "fixTypoMode": "FIX_TYPO_MODE_ON",
            },
            "sortSpec": {
                "sortMode": sort_mode_map[self.sorting_rule],
                "sortOrder": sort_order_map[self.sorting_time_order],
            },
            "groupSpec": {
                "groupMode": group_mode_map[self.grouping_method],
                "groupsOnPage": self.groups_on_page,
                "docsInGroup": self.docs_in_group,
            },
            "maxPassages": self.passages_count,
            "region": str(self.search_region_id),
            "l10n": l10n_map[self.notification_language],
            "folderId": self.folder_id.get_secret_value(),
            "responseFormat": "FORMAT_XML",
        }

    def _parse_results(self, results: str) -> List[Dict[str, Any]]:
        try:
            from parsel import Selector
        except ImportError as e:
            raise ImportError(
                "Could not import parsel. Install with `pip install parsel`."
            ) from e

        selector = Selector(text=results)
        tag_pattern = re.compile(r"</?.*?>")
        docs: List[Dict[str, Any]] = []

        error = selector.xpath("//error/text()").get()
        if error:
            raise RuntimeError(error)

        for doc in selector.xpath("//doc"):
            doc_id = doc.xpath("./@id").get()
            if not doc_id:
                continue

            title = doc.xpath("./title").get(default="")
            headline = doc.xpath("./headline").get(default="")
            passages = doc.xpath("./passages//passage").getall()

            if title:
                title = tag_pattern.sub("", title)
            if headline:
                headline = tag_pattern.sub("", headline)
            if passages:
                passages = [tag_pattern.sub("", passage) for passage in passages]

            modified_at = doc.xpath("./modtime/text()").get()
            url = doc.xpath("./url/text()").get()
            saved_copy_url = doc.xpath("./saved-copy-url/text()").get()

            if passages:
                content = self.passages_delimiter.join(passages)
                content_type = "passages"
            else:
                content = headline
                content_type = "headline"

            docs.append(
                {
                    "modified_at": modified_at,
                    "title": title,
                    "headline": headline,
                    "passages": passages,
                    "url": url,
                    "saved_copy_url": saved_copy_url,
                    "content": content,
                    "content_type": content_type,
                }
            )

        answer_docs = []
        for document in docs:
            answer_docs.append(
                {
                    field: document[field]
                    for field in self.answer_fields
                    if document.get(field)
                }
            )

        return answer_docs

    @staticmethod
    def _extract_error_message(response: "httpx.Response") -> str:
        try:
            payload = response.json()
        except Exception:
            return (response.text or "")[:500]

        if isinstance(payload, dict):
            for key in ("message", "error", "description"):
                value = payload.get(key)
                if value:
                    return str(value)
            return json.dumps(payload, ensure_ascii=False)

        return str(payload)

    def _raise_search_error(self, response: "httpx.Response") -> None:
        if not response.is_error:
            return

        details = self._extract_error_message(response)

        hint = ""
        if response.status_code == 401:
            hint = "Check YANDEX_API_KEY value and Authorization header format."
        elif response.status_code == 403:
            hint = (
                "Access denied. Ensure your service account/API key has "
                "`search-api.webSearch.user` on this folder and key scope `yc.search-api.execute`."
            )
        elif response.status_code == 400:
            hint = "Request payload is invalid for Yandex Search API v2."

        raise RuntimeError(
            f"Yandex Search API request failed ({response.status_code}): {details}. {hint}".strip()
        )

    @staticmethod
    def _extract_raw_data(payload: Dict[str, Any]) -> str | None:
        raw_data = payload.get("rawData")
        if raw_data:
            return raw_data

        response_obj = payload.get("response")
        if isinstance(response_obj, dict):
            return response_obj.get("rawData")

        return None

    def _decode_raw_xml(self, response: "httpx.Response") -> str:
        try:
            payload = response.json()
        except Exception as e:
            raise RuntimeError(
                "Yandex Search API v2 returned a non-JSON response; expected JSON with rawData."
            ) from e

        if not isinstance(payload, dict):
            raise RuntimeError("Unexpected Yandex Search API response format.")

        raw_data = self._extract_raw_data(payload)
        if not raw_data:
            details = self._extract_error_message(response)
            raise RuntimeError(
                "Yandex Search API v2 response does not contain rawData. "
                f"Details: {details}"
            )

        try:
            return base64.b64decode(raw_data).decode("utf-8", errors="replace")
        except Exception as e:
            raise RuntimeError("Could not decode Base64 rawData from Yandex Search API.") from e

    def _search(self, query: str) -> str:
        payload = self.base_payload
        payload["query"] = {**payload["query"], "queryText": query}
        headers = {"Authorization": f"Api-Key {self.api_key.get_secret_value()}"}

        try:
            import httpx
        except ImportError as e:
            raise ImportError("Could not import httpx. Install with `pip install httpx`.") from e

        with httpx.Client(base_url=self.base_url, timeout=self.request_timeout_s) as client:
            response = client.post(self.endpoint, json=payload, headers=headers)

        self._raise_search_error(response)
        return self._decode_raw_xml(response)

    async def _asearch(self, query: str) -> str:
        payload = self.base_payload
        payload["query"] = {**payload["query"], "queryText": query}
        headers = {"Authorization": f"Api-Key {self.api_key.get_secret_value()}"}

        try:
            import httpx
        except ImportError as e:
            raise ImportError("Could not import httpx. Install with `pip install httpx`.") from e

        async with httpx.AsyncClient(base_url=self.base_url, timeout=self.request_timeout_s) as client:
            response = await client.post(self.endpoint, json=payload, headers=headers)

        self._raise_search_error(response)
        return self._decode_raw_xml(response)

    def raw_results(self, query: str) -> str:
        return self._search(query)

    def results(self, query: str) -> List[Dict[str, Any]]:
        return self._parse_results(self.raw_results(query))

    async def raw_results_async(self, query: str) -> str:
        return await self._asearch(query)

    async def results_async(self, query: str) -> List[Dict[str, Any]]:
        return self._parse_results(await self.raw_results_async(query))


class YandexSearchAPIRetriever(BaseRetriever):
    """Yandex Search retriever for LangChain pipelines."""

    api_wrapper: YandexSearchAPIWrapper = Field(default_factory=YandexSearchAPIWrapper)
    k: int = 10

    @staticmethod
    def _generate_documents(results: List[Dict[str, Any]]) -> List[Document]:
        docs = []
        for result in results:
            doc = Document(
                page_content=result.pop("content", ""),
                metadata=result,
            )
            docs.append(doc)
        return docs

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> List[Document]:
        results = self.api_wrapper.results(query)
        docs = self._generate_documents(results)
        return docs[: self.k]

    async def _aget_relevant_documents(
        self,
        query: str,
        *,
        run_manager: AsyncCallbackManagerForRetrieverRun,
    ) -> List[Document]:
        results = await self.api_wrapper.results_async(query)
        docs = self._generate_documents(results)
        return docs[: self.k]


In [9]:
api_wrapper = YandexSearchAPIWrapper()
retriever = YandexSearchAPIRetriever(api_wrapper=api_wrapper, k=10)

print('Wrapper initialized:', type(api_wrapper).__name__)
print('Retriever initialized:', type(retriever).__name__)


Wrapper initialized: YandexSearchAPIWrapper
Retriever initialized: YandexSearchAPIRetriever


In [10]:
query = 'Latest updates about large language models in 2026'
results = api_wrapper.results(query)

print('Results found:', len(results))
for i, item in enumerate(results[:3], start=1):
    print(f'#{i}')
    print('url:', item.get('url'))
    content = item.get('content', '')
    print('content preview:', (content[:220] + '...') if len(content) > 220 else content)
    print('-' * 80)


RuntimeError: Yandex Search API request failed (403): Permission to [resource-manager.folder b1g2soqne7l6mr0gpt3o, resource-manager.cloud b1gp9pfkl03nj1vlt53v, organization-manager.organization bpf9imol1072iqana34n] denied. Access denied. Ensure your service account/API key has `search-api.webSearch.user` on this folder and key scope `yc.search-api.execute`.

### Exercise 1: Tune Search Configuration

Task:
1. Change `search_region_id`, `sorting_rule`, and `passages_count` in `YandexSearchAPIWrapper`.
2. Re-run the query cell.
3. Compare relevance and snippet quality.

Success criteria:
- You can explain which configuration gave the best result for your query type.


## Part 2. Retriever-Based QA with Sources


In [ ]:
def format_docs(docs):
    return '\\n\\n'.join(doc.page_content for doc in docs)


def generate_final_answer(response):
    template = (
        '### Question\n{question}\n\n'
        '### Answer\n{answer}\n\n'
        '### Sources\n{sources}'
    )

    sources = []
    for doc in response['context']:
        url = doc.metadata.get('url', 'unknown-url')
        snippet = (doc.page_content or '').strip().replace('\n', ' ')
        if len(snippet) > 220:
            snippet = snippet[:220] + '...'
        sources.append(f'- {snippet} ({url})')

    return template.format(
        question=response['question'],
        answer=response['answer'],
        sources='\n'.join(sources) if sources else '- No sources',
    )


rag_model = LC_GigaChat(
    model='GigaChat-Pro',
    credentials=os.getenv('GIGACHAT_CREDENTIALS'),
    verify_ssl_certs=False,
    streaming=False,
)

QA_TEMPLATE = (
    'Answer the question using only the provided context.\n'
    'If context is insufficient, say that explicitly.\n\n'
    'Context:\n{context}\n\n'
    'Question: {question}\n'
    'Answer:'
)

prompt = ChatPromptTemplate.from_template(QA_TEMPLATE)
output_parser = StrOutputParser()

chain_without_sources = (
    RunnableParallel(
        {
            'context': itemgetter('context') | RunnableLambda(format_docs),
            'question': itemgetter('question'),
        }
    )
    | prompt
    | rag_model
    | output_parser
)

chain_with_sources = RunnableParallel(
    {
        'context': itemgetter('question') | retriever,
        'question': itemgetter('question'),
    }
).assign(answer=chain_without_sources)



In [ ]:
query = 'Who won the Formula-1 Baku race in 2025?'
response = chain_with_sources.invoke({'question': query})
final_answer = generate_final_answer(response)
display(Markdown(final_answer))


### Exercise 2: Reduce Hallucinations

Task:
1. Modify `QA_TEMPLATE` to enforce stronger groundedness.
2. Test with a difficult query where retrieval may be noisy.
3. Compare answers before and after the prompt change.

Success criteria:
- Final answer clearly indicates uncertainty when context is weak.


## Part 3. Function Calling Agent with Yandex Search Tool

Now we expose Yandex Search as an agent tool (function-calling style) using `@giga_tool`.


In [ ]:
@giga_tool(
    few_shot_examples=[
        {
            'request': 'Find latest news about AI regulation in Europe',
            'params': {'query': 'latest news AI regulation Europe', 'top_k': 4},
        },
        {
            'request': 'Search for recent Rust language updates',
            'params': {'query': 'recent Rust language updates', 'top_k': 3},
        },
    ]
)
def yandex_web_search(
    query: str = Field(description='Web search query in natural language'),
    top_k: int = Field(default=5, description='Number of results to return (1-10)'),
) -> str:
    """Search the web via Yandex Search API and return concise snippets with URLs."""
    k = max(1, min(int(top_k), 10))
    docs = api_wrapper.results(query)[:k]

    if not docs:
        return 'No search results found.'

    lines = []
    for idx, doc in enumerate(docs, start=1):
        url = doc.get('url', 'unknown-url')
        content = (doc.get('content') or '').strip().replace('\n', ' ')
        if len(content) > 240:
            content = content[:240] + '...'
        lines.append(f'{idx}. {content} | {url}')

    return '\n'.join(lines)


In [ ]:
agent_model = LC_GigaChat(
    model='GigaChat-2-Max',
    credentials=os.getenv('GIGACHAT_CREDENTIALS'),
    verify_ssl_certs=False,
    streaming=False,
)

search_tools = [yandex_web_search]
search_agent = create_react_agent(
    agent_model.bind_functions(search_tools),
    search_tools,
    checkpointer=MemorySaver(),
    state_modifier=(
        'You are a search assistant. '
        'Always call yandex_web_search for fresh factual questions, then answer briefly with source URLs.'
    ),
)


In [ ]:
def run_search_agent_once(user_prompt: str, thread_id: str = 'yandex-demo'):
    response = search_agent.invoke(
        {'messages': [HumanMessage(content=user_prompt)]},
        config={'configurable': {'thread_id': thread_id}},
    )

    print(f'User: {user_prompt}')
    for msg in response['messages']:
        tool_calls = getattr(msg, 'tool_calls', None)
        if tool_calls:
            for tool_call in tool_calls:
                print(f"🔧 Tool: {tool_call['name']} | Args: {tool_call['args']}")

    print('Agent:', response['messages'][-1].content)


In [ ]:
run_search_agent_once('What are the latest official announcements about Python 3.14?')


## Part 4. Interactive Dialog via `input()`


In [ ]:
def chat_with_search_agent(thread_id: str = 'yandex-chat'):
    print("Interactive mode started. Type 'quit' to stop.")

    while True:
        user_input = input('User: ').strip()
        if user_input.lower() in {'quit', 'exit', ''}:
            print('Session finished.')
            break

        response = search_agent.invoke(
            {'messages': [HumanMessage(content=user_input)]},
            config={'configurable': {'thread_id': thread_id}},
        )

        for msg in response['messages']:
            tool_calls = getattr(msg, 'tool_calls', None)
            if tool_calls:
                for tool_call in tool_calls:
                    print(f"🔧 Tool: {tool_call['name']} | Args: {tool_call['args']}")

        print('Agent:', response['messages'][-1].content)


In [ ]:
# Run for live dialog
chat_with_search_agent()


### Exercise 3: Better Tool Routing

Task:
1. Improve the `state_modifier` instruction so the agent avoids unnecessary tool calls.
2. Test with both simple and time-sensitive prompts.

Success criteria:
- Tool is called for fresh facts, but skipped for obvious static questions.


### Exercise 4: Build a Search QA Assistant

Build your final assistant by editing only:
1. Tool description in `yandex_web_search`.
2. Agent `state_modifier`.
3. A short response policy (length, tone, citation format).

Validation checklist:
- Answers include source URLs.
- Agent handles uncertain/noisy results safely.
- Multi-turn memory works in interactive mode.


## Debrief

1. What prompt patterns most reliably triggered the search tool?
2. What was the main failure mode: wrong call, wrong query, or weak synthesis?
3. Which change improved output quality most: retrieval config, tool schema, or agent prompt?
